In [0]:
dbutils.widgets.text("catalog", "scl")
catalog = dbutils.widgets.get("catalog")

In [0]:
# CHECK 1 - All 8 source tables ingested into bronze 

from pyspark.sql import functions as F

dq_results = []

source_tables = ["customer", "orders", "lineitem", "part", "partsupp", "supplier", "nation", "region"]

ingest_rows = []
for t in source_tables:
    src_cnt = spark.table(f"samples.tpch.{t}").count()
    brz_cnt = spark.table(f"{catalog}.bronze.{t}").count()
    status = "PASS" if brz_cnt > 0 and brz_cnt == src_cnt else "FAIL"
    ingest_rows.append((t, int(src_cnt), int(brz_cnt), int(src_cnt - brz_cnt), status))
    dq_results.append((f"1. Bronze ingestion - {t}", f"bronze={brz_cnt:,} vs source={src_cnt:,}", status))

display(
    spark.createDataFrame(ingest_rows, ["table", "source_rows", "bronze_rows", "row_diff", "status"])
)

n_pass = sum(1 for r in ingest_rows if r[4] == "PASS")
print(f"Ingestion check: {n_pass}/8 source tables ingested completely")



In [0]:
# CHECK 2 - Primary keys populated (NOT NULL) and unique in silver tables, gold dims and fact_sales

pk_specs = [
    ("silver", "region",        ["r_regionkey"]),
    ("silver", "nation",        ["n_nationkey"]),
    ("silver", "customer",      ["c_custkey"]),
    ("silver", "supplier",      ["s_suppkey"]),
    ("silver", "orders",        ["o_orderkey"]),
    ("silver", "part",          ["p_partkey"]),
    ("silver", "partsupp",      ["ps_partkey", "ps_suppkey"]),
    ("silver", "lineitem",      ["l_orderkey", "l_linenumber"]),
    ("gold",   "dim_customer",  ["customer_key"]),
    ("gold",   "dim_supplier",  ["supplier_key"]),
    ("gold",   "dim_product",   ["part_key"]),
    ("gold",   "dim_geography", ["nation_key"]),
    ("gold",   "fact_sales",    ["order_key", "line_number"]),
]

pk_rows = []
for layer, tbl, keys in pk_specs:
    df = spark.table(f"{catalog}.{layer}.{tbl}")
    null_cond = " OR ".join([f"{c} IS NULL" for c in keys])
    row_cnt = df.count()
    null_cnt = df.filter(null_cond).count()
    distinct_cnt = df.select(keys).distinct().count()
    populated = null_cnt == 0
    unique = distinct_cnt == row_cnt
    status = "PASS" if populated and unique else "FAIL"
    pk_rows.append((layer, tbl, "+".join(keys), int(row_cnt), int(null_cnt), int(distinct_cnt), str(populated), str(unique), status))
    dq_results.append((f"2. PK populated & unique - {layer}.{tbl}", f"rows={row_cnt:,}, null_keys={null_cnt:,}, distinct_keys={distinct_cnt:,}", status))

display(
    spark.createDataFrame(pk_rows, ["layer", "table", "primary_key", "row_count", "null_key_count", "distinct_key_count", "pk_populated", "pk_unique", "status"])
)

n_pass = sum(1 for r in pk_rows if r[8] == "PASS")
print(f"Primary key check: {n_pass}/{len(pk_rows)} tables have populated, unique primary keys")

## 3. Fact table surrogate keys
`fact_sales` surrogate keys must be populated, and every FK must resolve to its silver source (no orphans).


In [0]:
# CHECK 3 - fact_sales surrogate keys populated (NOT NULL) and FKs resolve to silver sources

fact = spark.table(f"{catalog}.gold.fact_sales")

sk_cols = ["order_key", "line_number", "part_key", "supplier_key", "customer_key"]

sk_rows = []
for c in sk_cols:
    null_cnt = fact.filter(F.col(c).isNull()).count()
    status = "PASS" if null_cnt == 0 else "FAIL"
    sk_rows.append((f"{c} NOT NULL", int(null_cnt), status))
    dq_results.append((f"3. Fact surrogate key populated - {c}", f"null keys={null_cnt:,}", status))

# FK referential integrity - no orphan keys in fact_sales
fk_specs = [
    ("order_key",    "o_orderkey", "orders"),
    ("part_key",     "p_partkey",  "part"),
    ("supplier_key", "s_suppkey",  "supplier"),
    ("customer_key", "c_custkey",  "customer"),
]

for fk, src_key, src_tbl in fk_specs:
    src_df = spark.table(f"{catalog}.silver.{src_tbl}")
    orphan_cnt = (
        fact.select(fk)
        .distinct()
        .join(src_df.select(F.col(src_key).alias(fk)), on=fk, how="left_anti")
        .count()
    )
    status = "PASS" if orphan_cnt == 0 else "FAIL"
    sk_rows.append((f"{fk} -> silver.{src_tbl}", int(orphan_cnt), status))
    dq_results.append((f"3. Fact FK integrity - {fk} -> silver.{src_tbl}", f"orphan keys={orphan_cnt:,}", status))

display(
    spark.createDataFrame(sk_rows, ["key_or_relationship", "violation_count", "status"])
)

n_pass = sum(1 for r in sk_rows if r[2] == "PASS")
print(f"Fact surrogate key check: {n_pass}/{len(sk_rows)} key and FK checks passed")

In [0]:
# CHECK 4 - Numeric measures satisfy the documented business rules


rules = {
    "quantity > 0": F.col("quantity").isNull() | (F.col("quantity") <= 0),
    "extended_price >= 0": F.col("extended_price").isNull() | (F.col("extended_price") < 0),
    "order_total_price >= 0": F.col("order_total_price").isNull() | (F.col("order_total_price") < 0),
    "discount in [0, 1]": F.col("discount").isNull() | ~F.col("discount").between(0, 1),
    "tax >= 0": F.col("tax").isNull() | (F.col("tax") < 0),
    "discount_amount = extended_price * discount": (
        F.abs(F.col("discount_amount") - F.col("extended_price") * F.col("discount")) > F.lit(0.01)),
    "net_revenue = extended_price * (1 - discount)": (
        F.abs(F.col("net_revenue") - F.col("extended_price") * (F.lit(1) - F.col("discount"))) > F.lit(0.01)),
    "tax_amount = net_revenue * tax": (
        F.abs(F.col("tax_amount") - F.col("net_revenue") * F.col("tax")) > F.lit(0.01)),
    "total_amount = net_revenue * (1 + tax)": (
        F.abs(F.col("total_amount") - F.col("net_revenue") * (F.lit(1) + F.col("tax"))) > F.lit(0.01)),
}

violation_counts = (
    fact.agg(
        *[F.sum(F.when(cond, 1).otherwise(0)).alias(f"v{i}") for i, cond in enumerate(rules.values())]
    )
    .collect()[0]
)

rule_rows = []
for i, name in enumerate(rules):
    cnt = int(violation_counts[f"v{i}"])
    status = "PASS" if cnt == 0 else "FAIL"
    rule_rows.append((name, cnt, status))
    dq_results.append((f"4. Business rule - {name}", f"violations={cnt:,}", status))

display(
    spark.createDataFrame(rule_rows, ["business_rule", "violation_count", "status"])
)

n_pass = sum(1 for r in rule_rows if r[2] == "PASS")
print(f"Business rule check: {n_pass}/{len(rule_rows)} rules satisfied")

## 5. Gold ↔ silver reconciliation
Gold revenue and row counts must reconcile with silver.


In [0]:
# CHECK 5 - Gold revenue and counts reconcile with silver

recon_rows = []

# 5a. Row counts: fact_sales vs silver.lineitem (1:1 grain - one row per order line)
gold_fact_rows = spark.table(f"{catalog}.gold.fact_sales").count()
silver_line_rows = spark.table(f"{catalog}.silver.lineitem").count()
status = "PASS" if gold_fact_rows == silver_line_rows else "FAIL"
recon_rows.append(("fact_sales rows vs silver.lineitem rows", float(silver_line_rows), float(gold_fact_rows), float(silver_line_rows - gold_fact_rows), status))
dq_results.append(("5. Recon - fact_sales vs lineitem rows", f"gold={gold_fact_rows:,} vs silver={silver_line_rows:,}", status))

# 5b. Net revenue: gold fact_sales vs silver formula
gold_rev = spark.table(f"{catalog}.gold.fact_sales").agg(F.sum("net_revenue")).collect()[0][0]
silver_rev = (
    spark.table(f"{catalog}.silver.lineitem")
    .agg(F.sum(F.col("l_extendedprice") * (F.lit(1) - F.col("l_discount"))).alias("net_revenue"))
    .collect()[0][0]
)
rev_diff = float(gold_rev) - float(silver_rev)
status = "PASS" if abs(rev_diff) < 0.01 else "FAIL"
recon_rows.append(("net_revenue: gold vs silver (sum)", float(silver_rev), float(gold_rev), rev_diff, status))
dq_results.append(("5. Recon - net revenue gold vs silver", f"gold={float(gold_rev):,.2f} vs silver={float(silver_rev):,.2f} (diff={rev_diff:,.2f})", status))

# 5c. Gold dims vs silver sources (row counts)
dim_recon = [
    ("dim_customer", "customer"),
    ("dim_supplier", "supplier"),
    ("dim_product", "part"),
    ("dim_geography", "nation"),
]
for dim, src in dim_recon:
    dim_cnt = spark.table(f"{catalog}.gold.{dim}").count()
    src_cnt = spark.table(f"{catalog}.silver.{src}").count()
    status = "PASS" if dim_cnt == src_cnt else "FAIL"
    recon_rows.append((f"gold.{dim} rows vs silver.{src} rows", float(src_cnt), float(dim_cnt), float(src_cnt - dim_cnt), status))
    dq_results.append((f"5. Recon - {dim} vs silver.{src}", f"gold={dim_cnt:,} vs silver={src_cnt:,}", status))

# 5d. Gold KPI view total revenue vs gold fact (view must not drift from its source)
view_rev = (
    spark.table(f"{catalog}.gold.vw_annual_sales_kpis")
    .agg(F.sum("net_revenue"))
    .collect()[0][0]
)
view_diff = float(view_rev) - float(gold_rev)
status = "PASS" if abs(view_diff) < 0.01 else "FAIL"
recon_rows.append(("vw_annual_sales_kpis revenue vs fact_sales", float(gold_rev), float(view_rev), view_diff, status))
dq_results.append(("5. Recon - KPI view vs fact_sales revenue", f"view={float(view_rev):,.2f} vs fact={float(gold_rev):,.2f} (diff={view_diff:,.2f})", status))

display(
    spark.createDataFrame(recon_rows, ["reconciliation", "silver_or_source_value", "gold_value", "difference", "status"])
)

n_pass = sum(1 for r in recon_rows if r[4] == "PASS")
print(f"Reconciliation check: {n_pass}/{len(recon_rows)} reconciliations passed")

In [0]:
# CHECK 6 - Rejected (quarantined) record counts per silver table, with reason breakdown

quarantine_tables = ["region", "nation", "customer", "supplier", "orders", "part", "partsupp", "lineitem"]

rej_rows = []
reason_rows = []
for t in quarantine_tables:
    accepted_cnt = spark.table(f"{catalog}.silver.{t}").count()
    q_df = spark.table(f"{catalog}.silver.{t}_quarantine")
    q_cnt = q_df.count()
    total_cnt = accepted_cnt + q_cnt
    reject_pct = round(100.0 * q_cnt / total_cnt, 2) if total_cnt else 0.0
    rej_rows.append((t, int(total_cnt), int(accepted_cnt), int(q_cnt), reject_pct))

    # Reason breakdown (only for tables that actually rejected rows)
    if q_cnt > 0 and "validation_error" in q_df.columns:
        for r in q_df.groupBy("validation_error").count().collect():
            reason_rows.append((t, r["validation_error"], int(r["count"])))

print("=== Rejected records per silver table ===")
display(
    spark.createDataFrame(rej_rows, ["table", "total_rows", "accepted_rows", "rejected_rows", "rejected_pct"])
)

print("=== Rejection reason breakdown ===")
if reason_rows:
    display(
        spark.createDataFrame(reason_rows, ["table", "validation_error", "rejected_rows"])
    )
else:
    print("No rejected records found.")

total_rejected = sum(r[3] for r in rej_rows)
dq_results.append(("6. Rejected records reported", f"{total_rejected:,} rows quarantined across 8 silver tables (see breakdown above)", "PASS"))
print(f"Total rejected rows: {total_rejected:,}")

## Final DQ summary
All checks in one PASS/FAIL table with overall verdict.


In [0]:
# FINAL DQ SUMMARY - all checks, one PASS/FAIL table

# Dedupe in case a check cell was re-run in this session (accumulator pattern)
dq_results = list(dict.fromkeys(dq_results))

summary_df = spark.createDataFrame(dq_results, ["check", "detail", "status"]).orderBy("check")
display(summary_df)

n_pass = sum(1 for _, _, s in dq_results if s == "PASS")
n_total = len(dq_results)
overall = "PASSED" if n_pass == n_total else "FAILED"

print(f"Overall data quality: {overall} ({n_pass}/{n_total} checks passed)")
if overall == "FAILED":
    print("\nFailed checks:")
    for c, d, s in dq_results:
        if s != "PASS":
            print(f"  - {c}: {d}")

#job

![](/Workspace/Users/sakshijain@cyntexa.com/IAP/job.png)

#genie Agent

![](/Workspace/Users/sakshijain@cyntexa.com/IAP/genieAgent.png)

#13.final business challenge

three key finding,each supported by a specific query result and metric value.

- 1 — Revenue concentration: a small fraction of customers drives the majority of net revenue
- 2 — Discount erosion: discounts remove a significant share of gross revenue
- 3 — Supply-chain concentration: supplier revenue is geographically skewed


Two Opportunities for Growth or Operational Improvement -

1. Discount Policy Optimization
2. Long-Tail Customer Activation - specific customer generates the net revenue ,remaing customers contribution is less.

 Two Risks or Areas Requiring Further Investigation

- Risk 1 — High Line-Item Rejection Rate (17.2% quarantined)
- Risk 2 — Flat Revenue Trajectory Across All Seven Years - not any net_revneue growth.

Evidence from `vw_annual_sales_kpis` - Annual net revenue is effectively stagnant


recommandation
- For findings - 
    1.Making customers more engaged
    2.discount should be limited on each product.
    3.Try to increase the network and cover more areas.

- For opportunity - 
    1.Discount should not be greater than 0.5
    2.increase the customer involvement or contribution.

- For risk - 
    1.during data entry,apply data validation so that data accuracy increases.
    2.implement some differnt approaches to growth like offers on products and some marketing.